# How routes change with the objective and with obstacles (256×256)

PASSAGE stores six routes per scene so that a planner can be tested on two kinds of change: a new
cost objective on the same terrain, and new obstacles under the same objective. This notebook
measures how far those changes move the optimal route. It works first on one scene, then on the 64
preview scenes of the validation split at 256×256.

Two distances compare routes cell by cell. The **average symmetric surface distance (ASSD)** is the
mean distance from each cell of one route to the nearest cell of the other, taken both ways and
divided by the length of the reference route. The **95th-percentile Hausdorff distance (H95)** is
the distance, in cells, that 95 % of both routes stay within. Both are zero for identical routes.

**Run it yourself.** From a checkout of the [code repository](https://github.com/ThalesGroup/passage), start Jupyter with `uv run --frozen jupyter lab`, open this notebook and point `dataset_path` to a local copy of [the dataset](https://huggingface.co/datasets/thalesgroup/passage). With `dataset_path = None`, rows are read from the Hub at the pinned `dataset_revision` instead. It reads 64 preview rows and runs in under a minute at small resolutions, a few minutes at 4096×4096.

In [ ]:
resolution = 256
split = "validation"
sample_id = None  # scene of Section 1; None takes the first scene whose routes span a quarter of the side
scenes = 64  # preview scenes used in Section 2
output_dir = "notebook-artifacts/256x256/metrics"
dataset_path = "dataset"  # local copy of the dataset; None reads the Hub at dataset_revision
dataset_repo = "thalesgroup/passage"
dataset_revision = "65f1e4f0a37d8a7b476ddf02d48fe542542924ea"  # verified PASSAGE v1.0 upload
project_path = None  # checkout root; None searches upwards from the working directory

In [ ]:
from pathlib import Path
import json  # noqa: F401
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np  # noqa: F401
import pandas as pd
from IPython.display import Markdown, display

# Locate the code checkout (pyproject.toml next to src/passage) without changing directory.
project_root = Path(project_path).resolve() if project_path else next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src/passage").is_dir()), Path.cwd())
for extra in (project_root / "src", project_root / "scripts"):
    if extra.is_dir() and str(extra) not in sys.path:
        sys.path.insert(0, str(extra))

from passage.notebook_reports import make_source, resolve_path, save_figure, write_report  # noqa: E402, F401

source = make_source(dataset_path, dataset_repo=dataset_repo, dataset_revision=dataset_revision,
                     project_root=project_root)
output_path = resolve_path(output_dir, project_root)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 30)


def relative(path):
    """Show saved files relative to the checkout, never as machine paths."""
    path = Path(path)
    try:
        return path.resolve().relative_to(project_root.resolve()).as_posix()
    except ValueError:
        return path.name


def show(frame, caption=None, floatfmt="{:,.3f}"):
    """Display a table with readable numbers and an optional caption."""
    if caption:
        display(Markdown(f"*{caption}*"))
    floats = list(frame.select_dtypes("float").columns)
    ints = list(frame.select_dtypes("integer").columns)
    others = [c for c in frame.columns if c not in floats and c not in ints]
    styler = frame.style.hide(axis="index")
    for columns, fmt in ((floats, floatfmt), (ints, "{:,}"), (others, None)):
        if columns:
            styler = styler.format(fmt, subset=columns, na_rep="—")
    display(styler)


# One chart style for every PASSAGE notebook: recessive axes, light grid, left-aligned titles.
INK, MUTED, GRID = "#243537", "#5c6765", "#e3e6e3"
OBJECTIVE_COLORS = {"elevation": "#1767ac", "slope_uphill": "#d56418", "energy": "#a32b8e"}
OBJECTIVE_LABELS = {"elevation": "Elevation", "slope_uphill": "Slope (uphill)", "energy": "Energy"}
SPLIT_COLORS = {"train": "#1f8a70", "calibration": "#b07d1a", "validation": "#3a62b5", "test": "#c0392b"}
REGIME_COLORS = {"free": "#238b45", "obstacles": "#1767ac"}
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9, "axes.edgecolor": "#8a9290", "axes.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": GRID, "grid.linewidth": 0.6, "axes.axisbelow": True, "axes.titlelocation": "left",
    "axes.titlesize": 10.5, "axes.titleweight": "semibold", "axes.titlecolor": INK, "axes.labelcolor": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "legend.frameon": False, "lines.linewidth": 1.6,
    "figure.dpi": 110, "savefig.dpi": 140, "pdf.fonttype": 42, "ps.fonttype": 42,
})


def figure_title(fig, title, subtitle=None):
    """Left-aligned title and subtitle in a band reserved at the top of the figure."""
    height = fig.get_figheight()
    fig.suptitle(title, x=0.01, y=1 - 0.1 / height, ha="left", va="top", fontsize=12, fontweight="semibold", color=INK)
    if subtitle:
        fig.text(0.01, 1 - 0.4 / height, subtitle, ha="left", va="top", fontsize=8.5, color=MUTED)
    fig.subplots_adjust(top=min(fig.subplotpars.top, 1 - 0.95 / height))


# The shared trajectory style of the paper: one terrain scale, white start circle and goal star,
# green solid routes on free terrain and blue short dashes when obstacles are blocked.
from trajectory_style import render_cost_comparison, draw_comparison, comparison_handles  # noqa: E402, F401
from passage.plotting import decode_scene, render_scene, terrain_colormap, OBSTACLE_COLOR  # noqa: E402, F401


def first_scene(resolution, split, min_share=0.25, min_coverage=0.05):
    """Selection rule fixed in advance: the first scene of the split whose six stored routes all span at least
    `min_share` of the grid side and whose obstacles cover at least `min_coverage` of the cells."""
    for candidate in source.iter_rows(resolution, split, columns=["id", "metadata"], samples=False):
        lengths = [variant["length"] for paths in candidate["metadata"]["paths"].values()
                   for variant in (paths["free"], paths["obstacles"])]
        coverage = candidate["metadata"]["obstacles"]["ratio_actual"]
        if min(lengths) >= min_share * resolution and coverage >= min_coverage:
            return candidate["id"]
    raise ValueError("No scene of the split meets the selection rule")

## 1. One scene

Figure 1 draws the six routes of one scene on a single map: colour gives the objective, solid lines
are free terrain and dashed lines respect the obstacles. Table 1 then measures the distance between
each pair of routes that differ by one factor only.

In [ ]:
from matplotlib import patheffects
from matplotlib.lines import Line2D
from trajectory_style import obstacle_style
from passage.notebook_reports import route_metrics_report

PAIR_LABELS = {"astar_elevation": "elevation", "astar_slope": "slope_uphill", "astar_energy": "energy"}
row = source.get_row(resolution, split, sample_id=sample_id or first_scene(resolution, split), samples=False)
elevation, obstacles, start, goal = decode_scene(row)
free_paths = {f"path_{s}_free": row[f"path_{s}_free"] for s in PAIR_LABELS}
fig = render_scene(elevation, obstacles, start, goal, free_paths, legend=False,
                   title=f"Scene {row['id']} · {resolution}×{resolution}")
ax = fig.axes[0]
halo = [patheffects.Stroke(linewidth=3.2, foreground="white", alpha=0.85), patheffects.Normal()]
for index, (solver, objective) in enumerate(PAIR_LABELS.items()):
    ax.lines[index].set_color(OBJECTIVE_COLORS[objective])
    route = np.asarray(row[f"path_{solver}_obstacles"])
    if len(route):
        ax.plot(route[:, 1], route[:, 0], color=OBJECTIVE_COLORS[objective], linewidth=1.6,
                linestyle=obstacle_style(index), dash_capstyle="butt", path_effects=halo, zorder=4)
handles = [Line2D([], [], color=OBJECTIVE_COLORS[o], lw=1.8, label=OBJECTIVE_LABELS[o]) for o in OBJECTIVE_COLORS]
handles += [Line2D([], [], color=MUTED, lw=1.6, label="Free terrain"),
            Line2D([], [], color=MUTED, lw=1.6, linestyle=(0, (3.2, 1.7)), label="With obstacles")]
fig.legend(handles=handles, loc="lower center", ncol=5, fontsize=8, frameon=False)
figures = save_figure(fig, output_path, f"metrics-{resolution}-{row['id']}")
plt.show()

one = route_metrics_report(row, resolution, split)
def describe(route):
    _, solver_a, solver_b, condition = route.split("_", 3)
    return OBJECTIVE_LABELS[PAIR_LABELS[f"{solver_a}_{solver_b}"]], condition
pairs = []
for c in one["comparisons"]:
    (obj_a, cond_a), (obj_b, cond_b) = describe(c["route"]), describe(c["reference_route"])
    change = "obstacles added" if obj_a == obj_b else "objective changed"
    pairs.append({"Change": change, "Route": f"{obj_a} · {cond_a}", "Compared with": f"{obj_b} · {cond_b}",
                  "ASSD": c["assd"], "H95 (cells)": c["hausdorff_p95_pixels"]})
pairs = pd.DataFrame(pairs).sort_values(["Change", "Route"])
show(pairs, "Table 1. Distances between routes of the same scene that differ by one factor.", "{:,.3f}")

*Figure 1. The six stored routes of one scene. Colour: objective. Solid: free terrain. Dashed: obstacles
blocked. Where routes share a corridor, the dashes are offset so that each stays visible.*

## 2. Across the validation preview scenes

One scene can mislead, so the same measures now run on the preview scenes of the validation split.
For each objective, the **obstacle detour** is the extra cost that blocking the obstacles forces,
relative to the free-terrain optimum. H95 is reported as a share of the grid side, so that the
values can be compared across resolutions.

In [ ]:
stats = []
for preview in source.iter_rows(resolution, split, columns=None, samples=True, limit=scenes):
    result = route_metrics_report(preview, resolution, split)
    cost = {(r["solver"], r["task_variant"]): r["stored_cost"] for r in result["routes"]}
    status = {(r["solver"], r["task_variant"]): r["stored_status"] for r in result["routes"]}
    for c in result["comparisons"]:
        (obj_a, cond_a), (obj_b, cond_b) = describe(c["route"]), describe(c["reference_route"])
        stats.append({"scene": preview["id"], "change": "obstacles added" if obj_a == obj_b else "objective changed",
                      "pair": f"{obj_a} vs {obj_b}" if obj_a != obj_b else obj_a,
                      "condition": cond_a if cond_a == cond_b else "free vs obstacles",
                      "assd": c["assd"], "h95_share": None if c["hausdorff_p95_pixels"] is None
                      else 100 * c["hausdorff_p95_pixels"] / resolution})
    for solver, objective in PAIR_LABELS.items():
        if status[(solver, "obstacles")] == "ok":
            stats.append({"scene": preview["id"], "change": "detour", "pair": OBJECTIVE_LABELS[objective],
                          "detour_pct": 100 * (cost[(solver, "obstacles")] / cost[(solver, "free")] - 1)})
stats = pd.DataFrame(stats)
n_scenes = stats.scene.nunique()

detour = stats[stats.change == "detour"]
moved = stats[stats.change == "obstacles added"]
table = pd.DataFrame({
    "Objective": [OBJECTIVE_LABELS[o] for o in PAIR_LABELS.values()],
    "Scenes": [int((detour.pair == OBJECTIVE_LABELS[o]).sum()) for o in PAIR_LABELS.values()],
    "Route changed by obstacles (%)": [100 * (moved[moved.pair == OBJECTIVE_LABELS[o]].assd > 0).mean()
                                       for o in PAIR_LABELS.values()],
    "Median detour (%)": [detour[detour.pair == OBJECTIVE_LABELS[o]].detour_pct.median() for o in PAIR_LABELS.values()],
    "Median detour when the cost changes (%)": [
        detour[(detour.pair == OBJECTIVE_LABELS[o]) & (detour.detour_pct > 1e-9)].detour_pct.median()
        for o in PAIR_LABELS.values()],
    "90th percentile detour (%)": [detour[detour.pair == OBJECTIVE_LABELS[o]].detour_pct.quantile(0.9)
                                   for o in PAIR_LABELS.values()],
})
show(table, f"Table 2. Effect of the obstacles on the optimal route, {n_scenes} validation preview scenes.", "{:,.2f}")

fig, (ax_detour, ax_h95) = plt.subplots(1, 2, figsize=(11, 3.9))
fig.subplots_adjust(top=0.78, wspace=0.25)
labels = [OBJECTIVE_LABELS[o] for o in PAIR_LABELS.values()]
parts = ax_detour.boxplot([detour[detour.pair == label].detour_pct for label in labels], tick_labels=labels,
                          widths=0.5, patch_artist=True, medianprops={"color": INK, "linewidth": 1.4},
                          flierprops={"markersize": 3, "markeredgecolor": MUTED})
for patch, objective in zip(parts["boxes"], PAIR_LABELS.values()):
    patch.set(facecolor=OBJECTIVE_COLORS[objective], alpha=0.35, edgecolor=OBJECTIVE_COLORS[objective])
ax_detour.set_yscale("symlog", linthresh=1)
ax_detour.set(ylabel="Extra cost from obstacles (%, symmetric log)", title="Obstacle detour")

groups = [("obstacles added", "Obstacles added"), ("objective changed", "Objective changed")]
data = [stats[(stats.change == key)].h95_share.dropna() for key, _ in groups]
parts = ax_h95.boxplot(data, tick_labels=[label for _, label in groups], widths=0.5, patch_artist=True,
                       medianprops={"color": INK, "linewidth": 1.4},
                       flierprops={"markersize": 3, "markeredgecolor": MUTED})
for patch, color in zip(parts["boxes"], (REGIME_COLORS["obstacles"], MUTED)):
    patch.set(facecolor=color, alpha=0.3, edgecolor=color)
ax_h95.set(ylabel="H95 (% of grid side)", title="How far the route moves")
figure_title(fig, "Effect of the obstacles and of the objective on the optimal route",
             f"{n_scenes} validation preview scenes · {resolution}×{resolution}")
figures += save_figure(fig, output_path, f"metrics-{resolution}-preview")
plt.show()

*Figure 2. Left: extra cost forced by the obstacles, per objective. Right: how far a route moves, as
the 95th-percentile Hausdorff distance in percent of the grid side, when obstacles are added (same
objective) or when the objective changes (same condition). The detour axis is linear up to 1 % and
logarithmic above.*

## Summary

In [ ]:
med_obs = stats[stats.change == "obstacles added"].h95_share.median()
med_obj = stats[stats.change == "objective changed"].h95_share.median()
display(Markdown("\n".join([
    f"- On {n_scenes} validation preview scenes at {resolution}×{resolution}, obstacles change the optimal route in "
    + ", ".join(f"{v:.0f} % ({k.lower()})" for k, v in zip(table.Objective, table["Route changed by obstacles (%)"]))
    + " of the scenes.",
    "- When the obstacles change the optimal cost, the median detour is " + ", ".join(
        f"{v:.2f} % ({k.lower()})" for k, v in zip(table.Objective, table["Median detour when the cost changes (%)"]))
    + "; over all scenes, including unchanged ones, the median is " + ", ".join(
        f"{v:.2f} %" for v in table["Median detour (%)"]) + ".",
    f"- The median H95 is {med_obs:.1f} % of the grid side when obstacles are added and {med_obj:.1f} % when "
    "the objective changes.",
])))
report = {"resolution": resolution, "split": split, "scene": row["id"], "pairs": pairs.to_dict("records"),
          "preview_scenes": int(n_scenes), "obstacle_effect": table.to_dict("records"),
          "median_h95_share": {"obstacles_added": med_obs, "objective_changed": med_obj},
          "figures": [relative(f) for f in figures]}

In [ ]:
report_file = write_report(report, output_path, f"metrics-{resolution}.json")
display(Markdown(f"The numbers of this run are saved in `{relative(report_file)}`, next to the figures."))

---
*MIT License · Copyright (c) 2026 THALES. The full licence text is in the `LICENSE` file of the repository.*